# 02 – Data Quality & Cleaning

**Input:** data/raw/meals_raw.csv (23,663 meals from fetch_mensa.py)

**Goal:** check data quality, decide which canteens to keep, and clean the data.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/meals_raw.csv")
df.shape

(23663, 12)

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23663 entries, 0 to 23662
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                23663 non-null  int64  
 1   name              23663 non-null  str    
 2   category          23663 non-null  str    
 3   notes             23663 non-null  str    
 4   canteen_id        23663 non-null  int64  
 5   canteen_name      23663 non-null  str    
 6   city              23663 non-null  str    
 7   date              23663 non-null  str    
 8   prices.students   19768 non-null  float64
 9   prices.employees  19768 non-null  float64
 10  prices.pupils     1331 non-null   float64
 11  prices.others     17996 non-null  float64
dtypes: float64(4), int64(2), str(6)
memory usage: 2.2 MB


In [3]:
df.head()

,id,name,category,notes,canteen_id,canteen_name,city,date,prices.students,prices.employees,prices.pupils,prices.others
0,20807890,Paniertes Seelachsfilet,Fischgericht,"['Dillsoße', 'Fingermöhren', 'Brokkoli', 'Kurk...",63,"Leipzig, Mensa am Park",Leipzig,2024-10-02,4.10,6.15,NaN,8.15
1,20807891,Frikadelle mit Käse überbacken,Fleischgericht,"['Gebratenes Putensteak', 'Letscho ungarische ...",63,"Leipzig, Mensa am Park",Leipzig,2024-10-02,3.70,5.95,NaN,7.85
2,20559265,Letscho ungarische Art,Gemüsebeilage,"['Antioxidationsmittel', 'Süßungsmitteln', 'Se...",63,"Leipzig, Mensa am Park",Leipzig,2024-10-02,0.75,1.35,NaN,1.75
3,20559266,Fingermöhren,Gemüsebeilage,"['Antioxidationsmittel', 'Süßungsmitteln', 'Se...",63,"Leipzig, Mensa am Park",Leipzig,2024-10-02,0.75,1.35,NaN,1.75
4,20559267,Karotten-Kürbis-Gemüse,Gemüsebeilage,"['Antioxidationsmittel', 'Süßungsmitteln', 'Se...",63,"Leipzig, Mensa am Park",Leipzig,2024-10-02,0.75,1.35,NaN,1.75


In [4]:
df.isna().sum()

id                      0
name                    0
category                0
notes                   0
canteen_id              0
canteen_name            0
city                    0
date                    0
prices.students      3895
prices.employees     3895
prices.pupils       22332
prices.others        5667
dtype: int64

## Coverage: does every canteen have enough data?
I requested one Wednesday per week from 2024-10-01 to 2026-10-01.

In [5]:
all_dates = pd.date_range("2024-10-01", "2026-10-01", freq="W-WED")
total_dates = len(all_dates)
print("Dates requested per canteen:", total_dates)

Dates requested per canteen: 105


In [6]:
coverage = (
    df.groupby(["city", "canteen_name"])["date"]
      .nunique()
      .reset_index(name="dates_with_meals")
)

coverage["coverage_pct"] = (coverage["dates_with_meals"] / total_dates * 100).round(1)
coverage.sort_values("coverage_pct")

,city,canteen_name,dates_with_meals,coverage_pct
31,Würzburg,"Würzburg, Mensa am Studentenhaus",3,2.9
12,Jena,Mensa Ernst-Abbe-Platz,20,19.0
13,Jena,Mensa Philosophenweg,20,19.0
26,Saarbrücken / Saarland,Cafeteria Hochschule für Musik Saar,37,35.2
6,Düsseldorf,Mensa Kamp-Lintfort,46,43.8
7,Düsseldorf,"Mensa Universitätsstraße, Düsseldorf",46,43.8
1,Berlin,Mensa TU Hardenbergstraße,47,44.8
0,Berlin,"Berlin, Wilhelmstadtschulen",64,61.0
11,Heidelberg,"Heidelberg, Triplex-Mensa am Uniplatz",65,61.9
22,Passau,Cafeteria Nikolakloster Passau,67,63.8


In [7]:
CITIES = ["Heidelberg", "Berlin", "Hamburg", "München", "Leipzig",
          "Würzburg", "Nürnberg", "Dresden", "Köln",
          "Mannheim", "Chemnitz", "Straubing", "Passau", "Regensburg",
          "Düsseldorf", "Stuttgart", "Saarbrücken / Saarland",
          "Dortmund / Nordrhein-Westfalen", "Jena"]

missing_cities = set(CITIES) - set(df["city"].unique())
missing_cities

{'Dortmund / Nordrhein-Westfalen', 'Hamburg', 'München'}

## Decision: which canteens do I keep?
Looking at the coverage table, there is a clear gap: 7 canteens are below 45%, and all others are above 60%.
Jena and Düsseldorf have identical counts for both canteens, which suggests their data stopped at some point.
Hamburg, München and Dortmund returned no data at all.
Two canteens are schools, not universities (Berlin Wilhelmstadtschulen, Straubing ABG Gymnasium).

My rule:
1. Keep canteens with at least 50% coverage, because it falls in the natural gap and removes canteens whose data stopped.
2. Remove school canteens, because this project is about university students.

In [8]:
MIN_COVERAGE = 50
SCHOOL_CANTEENS = ["Berlin, Wilhelmstadtschulen",
                   "Straubing, ABG Anton-Bruckner-Gymnasium"]

good_canteens = coverage[coverage["coverage_pct"] >= MIN_COVERAGE]["canteen_name"]
df_clean = df[df["canteen_name"].isin(good_canteens)]
df_clean = df_clean[~df_clean["canteen_name"].isin(SCHOOL_CANTEENS)]

print("Canteens kept:", df_clean["canteen_name"].nunique())
print("Cities kept:", df_clean["city"].nunique())
print("Meals kept:", len(df_clean))

Canteens kept: 23
Cities kept: 13
Meals kept: 17918


## Part B – Cleaning
Problems noted in notebook 01:
1. Text has stray line breaks (\n)
2. Notes are stored as text that looks like a list
3. prices.pupils is almost always empty
4. Some meals are priced per 100g, not per portion
5. Some meals have no student price

In [9]:
df_clean = df_clean.copy()

In [10]:
df_clean["name"] = df_clean["name"].str.replace("\n", " ").str.strip()
df_clean["category"] = df_clean["category"].str.strip()
df_clean["name"].head()

0           Paniertes Seelachsfilet
1    Frikadelle mit Käse überbacken
2            Letscho ungarische Art
3                      Fingermöhren
4            Karotten-Kürbis-Gemüse
Name: name, dtype: str

In [11]:
print(type(df_clean["notes"].iloc[0]))
print(df_clean["notes"].iloc[0])

<class 'str'>
['Dillsoße', 'Fingermöhren', 'Brokkoli', 'Kurkumareis', 'glutenhaltiges Getreide', 'Fisch', 'Weizen', 'Eier', 'Milch/ Milchzucker', 'Sulfit/ Schwefeldioxid', 'Fisch', 'Antioxidationsmittel']


In [12]:
import ast

df_clean["notes"] = df_clean["notes"].apply(ast.literal_eval)                        # text → real list
df_clean["notes"] = df_clean["notes"].apply(lambda lst: "; ".join(n.strip() for n in lst))  # list → clean text
df_clean["notes"].head()

0    Dillsoße; Fingermöhren; Brokkoli; Kurkumareis;...
1    Gebratenes Putensteak; Letscho ungarische Art;...
2    Antioxidationsmittel; Süßungsmitteln; Senf; Vegan
3    Antioxidationsmittel; Süßungsmitteln; Senf; Vegan
4    Antioxidationsmittel; Süßungsmitteln; Senf; Vegan
Name: notes, dtype: str

In [13]:
df_clean = df_clean.drop(columns=["prices.pupils"])
df_clean.columns

Index(['id', 'name', 'category', 'notes', 'canteen_id', 'canteen_name', 'city',
       'date', 'prices.students', 'prices.employees', 'prices.others'],
      dtype='str')

In [14]:
per_100g = df_clean["name"].str.contains(r"100\s?g", case=False, regex=True)

print("Meals priced per 100g:", per_100g.sum())
df_clean[per_100g][["canteen_name", "name", "prices.students"]].head(10)

Meals priced per 100g: 417


,canteen_name,name,prices.students
3003,"Dresden, Alte Mensa",100g gebrannte Mandeln,3.10
9475,"Heidelberg, Mensa Im Neuenheimer Feld 304",Schlemmerbuffet (je 100g) Reichhaltige Auswahl...,0.92
9480,"Heidelberg, Mensa Im Neuenheimer Feld 304",Schlemmerbuffet (je 100g) Reichhaltige Auswahl...,0.92
9486,"Heidelberg, Mensa Im Neuenheimer Feld 304",Schlemmerbuffet (je 100g) Reichhaltige Auswahl...,0.92
9492,"Heidelberg, Mensa Im Neuenheimer Feld 304",Schlemmerbuffet (je 100g) Reichhaltige Auswahl...,0.92
9498,"Heidelberg, Mensa Im Neuenheimer Feld 304",Schlemmerbuffet (je 100g) Reichhaltige Auswahl...,0.92
9504,"Heidelberg, Mensa Im Neuenheimer Feld 304",Schlemmerbuffet (je 100g) Reichhaltige Auswahl...,0.92
9510,"Heidelberg, Mensa Im Neuenheimer Feld 304",Schlemmerbuffet (je 100g) Reichhaltige Auswahl...,0.92
9516,"Heidelberg, Mensa Im Neuenheimer Feld 304",Schlemmerbuffet (je 100g) Reichhaltige Auswahl...,0.92
9522,"Heidelberg, Mensa Im Neuenheimer Feld 304",Schlemmerbuffet (je 100g) Reichhaltige Auswahl...,0.92


In [15]:
df[df.index.isin(per_100g[per_100g].index)]["canteen_name"].value_counts()

canteen_name
Köln, Mensa Zülpicher Straße                 162
Heidelberg, Mensa Im Neuenheimer Feld 304     95
Köln, Mensa Südstadt                          94
Heidelberg, Triplex-Mensa am Uniplatz         65
Dresden, Alte Mensa                            1
Name: count, dtype: int64

In [16]:
removed = df.loc[per_100g[per_100g].index]

removed.groupby("canteen_name")["prices.students"].median().sort_values()

canteen_name
Köln, Mensa Südstadt                         0.75
Köln, Mensa Zülpicher Straße                 0.75
Heidelberg, Mensa Im Neuenheimer Feld 304    0.92
Heidelberg, Triplex-Mensa am Uniplatz        0.99
Dresden, Alte Mensa                          3.10
Name: prices.students, dtype: float64

In [17]:
removed = removed.copy()
removed["in_name"] = removed["name"].str.contains(r"100\s?g", case=False, regex=True)
removed.groupby("canteen_name")["in_name"].mean().round(2)

canteen_name
Dresden, Alte Mensa                          1.0
Heidelberg, Mensa Im Neuenheimer Feld 304    1.0
Heidelberg, Triplex-Mensa am Uniplatz        1.0
Köln, Mensa Südstadt                         1.0
Köln, Mensa Zülpicher Straße                 1.0
Name: in_name, dtype: float64

In [18]:
df_clean = df_clean[~per_100g]

In [19]:
no_price = df_clean[df_clean["prices.students"].isna()]
print("Meals without student price:", len(no_price))
no_price["canteen_name"].value_counts()

Meals without student price: 258


canteen_name
Mensa / Mensacafe Saarbrücken                110
Chemnitz, Reichenhainer Straße, Mensa         70
Heidelberg, Triplex-Mensa am Uniplatz         29
Chemnitz, Straße der Nationen, Mensa          19
Heidelberg, Mensa Im Neuenheimer Feld 304     13
Köln, Mensa Zülpicher Straße                   5
Mensa Insel Schütt                             4
Mensa Regensburger Straße                      4
Dresden, Alte Mensa                            3
Köln, Mensa Südstadt                           1
Name: count, dtype: int64

In [20]:
no_price[["canteen_name", "name", "category"]].sample(10, random_state=1)

,canteen_name,name,category
12362,"Chemnitz, Reichenhainer Straße, Mensa","Neues Jahr, neues Glück“ - ""Schlag das StuWe!”...",zZz Hinweis zZz
8154,Mensa / Mensacafe Saarbrücken,Sommerhilfe,Information
12368,"Chemnitz, Reichenhainer Straße, Mensa","""Save the Date"" Nach dem Quiz ist vor dem Quiz...",zZz Hinweis zZz
12917,"Chemnitz, Straße der Nationen, Mensa",Die Mensa und Cafeteria Straße der Nationen bl...,Achtung
12669,"Chemnitz, Reichenhainer Straße, Mensa",Wir wünschen allen frohe Weihnachten und einen...,Frohe Festtage
12354,"Chemnitz, Reichenhainer Straße, Mensa",Das Team der mensa55 wünscht euch eine tolle W...,Hinweis
10398,"Heidelberg, Triplex-Mensa am Uniplatz",Eis,A
8097,Mensa / Mensacafe Saarbrücken,Sommerhilfe,Information
10330,"Heidelberg, Triplex-Mensa am Uniplatz",Pasta + More,Pasta Container
12383,"Chemnitz, Reichenhainer Straße, Mensa","Neues Jahr, neues Glück“ - ""Schlag das StuWe!”...",zZz Hinweis zZz


In [21]:
df_clean = df_clean[df_clean["prices.students"].notna()]

In [22]:
df_clean["date"] = pd.to_datetime(df_clean["date"])
df_clean.dtypes

id                           int64
name                           str
category                       str
notes                          str
canteen_id                   int64
canteen_name                   str
city                           str
date                datetime64[us]
prices.students            float64
prices.employees           float64
prices.others              float64
dtype: object

In [23]:
print("Final rows:", len(df_clean))
print("Canteens:", df_clean["canteen_name"].nunique())
df_clean.isna().sum()

Final rows: 17243
Canteens: 23


id                     0
name                   0
category               0
notes                  0
canteen_id             0
canteen_name           0
city                   0
date                   0
prices.students        0
prices.employees       0
prices.others       1771
dtype: int64

In [24]:
df_clean.to_csv("../data/processed/meals_clean.csv", index=False)

## Cleaning decisions
- **Per-100g meals:** My first rule searched the name and notes for "100g" and found 985 meals.
  Checking showed that in Stuttgart, Mannheim and Freiburg "100g" only appeared in nutrition notes
  ("Nährwerte pro 100 g"), not in the price. So I changed the rule to search the name only.
  Result: removed 417 meals priced per 100g (Heidelberg and Köln buffets, snacks in Dresden),
  because they can't be compared with per-portion prices.
- **No student price:** Removed 258 rows. Most were not meals but announcements posted as menu
  items (categories like "Information", "zZz Hinweis zZz", "Frohe Festtage"). A few real items were lost too, which I accept.
- **Text:** Removed line breaks (\n) from names and categories, and turned notes into readable text.
- **Columns:** Dropped